##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [1]:
import os
import cv2
import random
import numpy as np
import tensorflow as tf

seed_constant = 23
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

image_height, image_width = 64, 64
max_images_per_class = 1000

dataset_directory = "UCF11_updated_mpg"
classes_list = ["basketball", "biking", "horse_riding"]

model_output_size = len(classes_list)

def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)

    if not video_reader.isOpened():
        return frames_list

    while True:
        success, frame = video_reader.read()
        if not success:
            break

        resized_frame = cv2.resize(frame, (image_width, image_height))
        normalized_frame = resized_frame / 255.0
        frames_list.append(normalized_frame)

    video_reader.release()
    return frames_list

def create_dataset():
    features = []
    labels = []

    for class_index, class_name in enumerate(classes_list):
        print(f'Extracting data of class: {class_name}')

        class_dir = os.path.join(dataset_directory, class_name)

        if not os.path.exists(class_dir):
            print(f"[WARNING] Folder not found: {class_dir}")
            continue

        class_count = 0

        for root, dirs, files in os.walk(class_dir):
            for file_name in files:
                if not file_name.lower().endswith((".mpg", ".avi", ".mp4", ".mov")):
                    continue

                video_file_path = os.path.join(root, file_name)
                frames = frames_extraction(video_file_path)

                if len(frames) == 0:
                    continue

                for frame in frames:
                    features.append(frame)
                    labels.append(class_index)
                    class_count += 1

                    if class_count >= max_images_per_class:
                        break

                if class_count >= max_images_per_class:
                    break

            if class_count >= max_images_per_class:
                break

        print(f"Collected {class_count} frames for {class_name}")

    features = np.asarray(features)
    labels = np.array(labels)

    return features, labels

features, labels = create_dataset()
print("Features shape:", features.shape)
print("Labels shape:", labels.shape)

Extracting data of class: basketball
Collected 1000 frames for basketball
Extracting data of class: biking
Collected 1000 frames for biking
Extracting data of class: horse_riding
Collected 1000 frames for horse_riding
Features shape: (3000, 64, 64, 3)
Labels shape: (3000,)


In [2]:
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# One-hot encoding
one_hot_encoded_labels = to_categorical(labels)

# Split
features_train, features_test, labels_train, labels_test = train_test_split(
    features,
    one_hot_encoded_labels,
    test_size=0.2,
    shuffle=True,
    random_state=23
)

print("Train shape:", features_train.shape)
print("Test shape:", features_test.shape)

Train shape: (2400, 64, 64, 3)
Test shape: (600, 64, 64, 3)


In [4]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

base_cnn = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(image_height, image_width, 3)
)

base_cnn.trainable = False

inputs = Input(shape=(image_height, image_width, 3))
x = base_cnn(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(model_output_size, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

model.summary()

history = model.fit(
    features_train,
    labels_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    shuffle=True,
    callbacks=callbacks
)

C:\Users\HP\AppData\Local\Temp\ipykernel_3776\1101621947.py:7: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_cnn = MobileNetV2(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 2, 2, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 12s 121ms/step - accuracy: 0.9682 - loss: 0.0875 - val_accuracy: 1.0000 - val_loss: 0.0024 - learning_rate: 0.0010
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 114ms/step - accuracy: 1.0000 - loss: 0.0015 - val_accuracy: 1.0000 - val_loss: 9.5467e-04 - learning_rate: 0.0010
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 105ms/step - accuracy: 1.0000 - loss: 5.7759e-04 - val_accuracy: 1.0000 - val_loss: 7.0367e-04 - learning_rate: 0.0010
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 110ms/step - accuracy: 1.0000 - loss: 5.3844e-04 - val_accuracy: 1.0000 - val_loss: 6.4791e-04 - learning_rate: 0.0010
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 1.0000 - loss: 3.4848e-04 - val_accuracy: 1.0000 - val_loss: 3.4820e-04 - learning_rate: 0.0010
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 96ms/step - accuracy: 1.0000 - loss: 2.2898e-04 - val_accuracy: 1.0000 - val_loss: 2.7470e-04 - learning_rate: 0.0010
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step 

In [5]:
test_loss, test_accuracy = model.evaluate(features_test, labels_test)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 111ms/step - accuracy: 1.0000 - loss: 4.4825e-05
Test Loss: 4.4825399527326226e-05
Test Accuracy: 1.0


In [19]:
student_name = "Razan Alkharasani" 
save_path = f"{student_name}_ucf11_model.h5"
model.save(save_path)
print(f"Model saved as {save_path}")

Model saved as Razan Alkharasani_ucf11_model.h5


In [17]:
import numpy as np
import cv2

def predict_video(video_file_path, num_frames_to_sample=20):
    predicted_labels_probabilities_np = np.zeros((num_frames_to_sample, model_output_size), dtype=float)

    video_reader = cv2.VideoCapture(video_file_path)

    if not video_reader.isOpened():
        print(f"Could not open video: {video_file_path}")
        return

    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(video_frames_count / num_frames_to_sample), 1)

    for frame_counter in range(num_frames_to_sample):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()

        if not success:
            continue

        resized_frame = cv2.resize(frame, (image_width, image_height))
        normalized_frame = resized_frame / 255.0

        predicted_labels_probabilities = model.predict(
            np.expand_dims(normalized_frame, axis=0),
            verbose=0
        )[0]

        predicted_labels_probabilities_np[frame_counter] = predicted_labels_probabilities

    video_reader.release()

    average_predicted_labels_probabilities = predicted_labels_probabilities_np.mean(axis=0)
    predicted_label = np.argmax(average_predicted_labels_probabilities)

    print("Video:", video_file_path)
    print("Predicted Class:", classes_list[predicted_label])
    print("Class Probabilities:", average_predicted_labels_probabilities)
    print("-" * 50)


In [23]:
predict_video("Youtube_Videos/basketball.mp4")
predict_video("Youtube_Videos/biking.mp4")
predict_video("Youtube_Videos/horse_riding.mp4")


Video: Youtube_Videos/basketball.mp4
Predicted Class: horse_riding
Class Probabilities: [2.32577118e-08 1.30475528e-01 8.69524475e-01]
--------------------------------------------------
Video: Youtube_Videos/biking.mp4
Predicted Class: biking
Class Probabilities: [2.45053696e-04 8.13762870e-01 1.85992087e-01]
--------------------------------------------------
Video: Youtube_Videos/horse_riding.mp4
Predicted Class: horse_riding
Class Probabilities: [0.00096661 0.22724711 0.77178628]
--------------------------------------------------


Instead of training a CNN from scratch as in the lab, a pretrained MobileNetV2 model was used to improve performance and reduce training time.

The model misclassified the basketball video as horse riding. This error may be due to differences between the training dataset and the external YouTube video, such as camera angle, background, lighting conditions, and motion patterns. The model was trained on a limited number of classes and specific video styles, which affects its ability to generalize to real-world videos.                          

The model performance can be improved by:

- Increasing the diversity of the training data to include more variations in camera angles and environments.
- Using video-level splitting instead of frame-level splitting to avoid data leakage.
- Applying data augmentation techniques such as flipping, rotation, and brightness changes.
- Training the model for more epochs or fine-tuning more layers of the pretrained network.
- Using more advanced architectures designed for video understanding.